Historical study notebook. Its original input is unavailable in this public edition, so this is not a runnable public demo. It may require an input with the original structure and dependencies. This work used supplied learning resources and starter material; retained attribution appears with the notebook content.



In [ ]:
!pip install -q PyPDF2 llama-cpp-python huggingface_hub

In [ ]:
import json, uuid
from llama_cpp import Llama, LlamaGrammar

llm = Llama.from_pretrained(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
    filename="mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    n_gpu_layers=-1,
    n_ctx=4096,
    seed=42,
    verbose=False,
)

# Fixed doc-type set (lowercase_snake_case, per the task)
DOC_TYPES = ["cover_letter", "certificate_of_quality", "packaging_specification",
             "bse_tse_declaration", "material_description", "supplier_qualification",
             "chain_of_custody", "unknown"]

# Grammar so the model can ONLY return one of these exact strings
_grammar_str = "root ::= " + " | ".join(f'"{t}"' for t in DOC_TYPES)
LABEL_GRAMMAR = LlamaGrammar.from_string(_grammar_str)

def label_call(prompt):
    out = llm(f"[INST] {prompt} [/INST]", max_tokens=20, temperature=0.0, grammar=LABEL_GRAMMAR)
    return out["choices"][0]["text"].strip()


In [ ]:
from PyPDF2 import PdfReader

def load_pdf(path, filename, year="2024", user_id="xyz"):
    reader = PdfReader(path)
    file_id = str(uuid.uuid4())
    store = []
    for i, page in enumerate(reader.pages):
        store.append({
            "file_id": file_id,
            "user_id": user_id,
            "doc_type": "unknown",       # filled in at Step 3
            "year": year,
            "filename": filename,
            "page_number": i + 1,
            "text": page.extract_text() or ""
        })
    return store

pdf_metadata_store = (
    load_pdf("your-document.pdf", "your-document.pdf")
+ load_pdf("your-document.pdf", "your-document.pdf")
)
print(f"Loaded {len(pdf_metadata_store)} pages from 2 PDFs")


In [ ]:
def classify_query_llm(query):
    prompt = f"""You are a query router for a pharmaceutical document system.
Given a user question, decide which document type most likely contains the answer.
Choose exactly ONE from: {DOC_TYPES}

Guidance:
- document identifier / document identifier / test results -> certificate_of_quality
- packaging materials / blister / carton -> packaging_specification
- animal origin / BSE / TSE compliance -> bse_tse_declaration
- materials of construction / sterilization -> material_description
- supplier audit / ISO certification -> supplier_qualification
- traceability / chain of custody / shipment -> chain_of_custody
- storage conditions / cover letter -> cover_letter
- if unclear -> unknown

User question: "{query}"
Respond with only the doc_type."""
    return label_call(prompt)


In [ ]:
def classify_doc_type_llm(text):
    if not text.strip():
        return "unknown"
    prompt = f"""Classify this pharmaceutical document page into exactly ONE type from: {DOC_TYPES[:-1]}
- cover_letter: formal letter (e.g. "To Whom It May Concern"), storage conditions
- certificate_of_quality: document identifier, manufacture/expiration dates, test results
- packaging_specification: packaging components, materials, document identifier
- bse_tse_declaration: animal-origin / spongiform encephalopathy compliance
- material_description: materials of construction, sterilization, physical properties
- supplier_qualification: supplier audits, ISO certs, approved product lists
- chain_of_custody: manufactured assemblies, traceability, shipment flow

Page text:
{text[:1500]}

Respond with only the snake_case doc_type."""
    return label_call(prompt)

# Tag every page
for page in pdf_metadata_store:
    page["doc_type"] = classify_doc_type_llm(page["text"])

# Quick look at what got assigned
for p in pdf_metadata_store:
    print(p["filename"], "p"+str(p["page_number"]), "->", p["doc_type"])


In [ ]:
def keyword_fallback(query, docs):
    terms = ["document identifier", "document identifier", "specification", "certificate",
             "lot", "sterilization", "bse", "supplier", "chain of custody"]
    hits = [d for d in docs if any(t in d["text"].lower() for t in terms)]
    return hits or docs

def route_query(query, use_fallback=True):
    predicted_doc_type = classify_query_llm(query)
    matched = [p for p in pdf_metadata_store if p["doc_type"] == predicted_doc_type]
    if not matched and use_fallback:
        matched = keyword_fallback(query, pdf_metadata_store)
    return {
        "query": query,
        "predicted_doc_type": predicted_doc_type,
        "matched_documents": matched
    }


In [ ]:
test_queries = [
    "What are the document identifier listed in this document?",
    "What are the BSE/TSE compliance dates?",
    "What packaging materials are specified?",
    "Who is the approved supplier and what certifications do they hold?",
    "Show me all the compliance documentation",   # deliberately vague
]

for q in test_queries:
    r = route_query(q)
    print(f"\nQ: {q}")
    print(f"   -> predicted: {r['predicted_doc_type']}, matched {len(r['matched_documents'])} page(s)")
    for d in r["matched_documents"][:3]:
        print(f"      {d['filename']} p{d['page_number']}")


In [ ]:
result = route_query("What are the document identifier listed in this document?")
print(json.dumps(result, indent=2))
